# Basic Binary T-Test Function
The aim of this notebook is to design and test the basic binary t-test function. This assumes there are two groups (treatment / control) of equal size (50:50), with assumed equal variances.

In [4]:
# Import the Necessary Packages
import pandas as pd
import numpy as np
from dataclasses import dataclass
from scipy import stats
from pathlib import Path

In [5]:
# Import the Example Data
df = pd.read_csv(Path("example_data.csv"))

# Filter to Critical Data
df = df[["User ID", "ExperiencedCondition", "complete"]].rename(
    columns={
        "User ID": "user_id",
        "ExperiencedCondition": "assignment",
        "complete": "complete",
    }
)

# Correct Column Types
df["complete"] = df["complete"].astype(int)
df["assignment"] = df["assignment"].astype(bool)
df.head(20)

,user_id,assignment,complete
0,172777,False,1
1,175658,True,0
2,175669,True,1
3,176151,True,1
4,176165,True,0
5,176168,True,1
6,176461,True,0
7,176486,True,0
8,176488,True,0
9,176494,True,0


In [6]:
# Define the Results Class
@dataclass(frozen=True)
class t_test_result:
    mean_control: float
    mean_treatment: float
    absolute_diff: float
    relative_diff: float
    se: float
    t_stat: float
    p_value: float
    ci_low: float
    ci_high: float
    relative_ci_low: float
    relative_ci_high: float

In [ ]:
def t_test_binary(
    df,
    metric="conversion_rate",
    assignment_col="assignment",
    alpha=0.05
):

    """
    This is a basic Welch's t-test function that takes raw unit-level data 
    with two groups of equal size, and a binary metric of interest. It needs
    the following arguements:

    df: A dataframe with one row per metric-arm, containing at least the
                columns named by metric_col and assignment
    metric: The metric used in the analysis, default here is conversion_rate
    assignment_col: Column of booleans identifying treatment (True) vs
                control (False) rows. Default is "assignment".
    alpha: The significance level (default is 0.05).

    Returns a t_test_result object (see t_test docstring
    for field definitions).

    Raises:
        KeyError: if any required column is missing.
        ValueError: if alpha is not strictly between 0 and 1; if the metric
                is not found, or found more than once, in either arm; if
                either arm's n is fewer than 2; if either arm's variance is
                zero or negative; or if mean_control is 0.
    """

    # Include Critical Error Checks
    if df.empty:
        raise ValueError("Dataframe is empty — no data to test")
    required = (assignment_col, metric)
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"df is missing required column(s): {missing}")
    if not 0 < alpha < 1:
        raise ValueError(f"Alpha must be between 0 and 1, got {alpha}")

    # Split the Datasets by Assignment
    treatment_values = df.loc[df[assignment_col], metric].to_numpy(dtype=float)
    control_values = df.loc[~df[assignment_col], metric].to_numpy(dtype=float)
 
    # Error Check for Missing Data
    if treatment_values.size == 0 or control_values.size == 0:
        raise ValueError(
            f"expected units in both arms, got {treatment_values.size}"
            f"treatment and {control_values.size} control"
        )
 
    # Compute the Summary Statistics That the Summary Version Is Handed
    n_treatment = treatment_values.size
    n_control = control_values.size
    mean_treatment = float(treatment_values.mean())
    mean_control = float(control_values.mean())
 
    # Variance Needs at Least 2 Observations per Group
    if n_treatment < 2 or n_control < 2:
        raise ValueError(
            f"each arm needs at least 2 observations to trust the variance "
            f"(got n_treatment={n_treatment}, n_control={n_control})"
        )
 
    # Estimate Variances
    var_treatment = (mean_treatment * (1 - mean_treatment) * n_treatment) / (n_treatment - 1)
    var_control = (mean_control * (1 - mean_control) * n_control) / (n_control - 1)
 
    # Error Check for Breaking Denominator / Degenerate Variance
    if mean_control == 0:
        raise ValueError(
            f"mean_control is 0 for {metric!r}; relative effects undefined"
        )
    if var_treatment <= 0:
        raise ValueError(f"{metric!r} has zero/negative variance in Treatment arm")
    if var_control <= 0:
        raise ValueError(f"{metric!r} has zero/negative variance in Control arm")
 
    # Calculate the Standard Error of the Treatment and Control Groups
    se = np.sqrt(var_treatment / n_treatment + var_control / n_control)
 
    # Calculate the Absolute and Relative Difference
    absolute_diff = mean_treatment - mean_control
    relative_diff = absolute_diff / mean_control
 
    # Calculate the Degrees of Freedom
    dof = (var_treatment / n_treatment + var_control / n_control) ** 2 / (
        (var_treatment / n_treatment) ** 2 / (n_treatment - 1)
        + (var_control / n_control) ** 2 / (n_control - 1)
    )
 
    # Run the t-test (raw arrays available, so hand it to scipy)
    t_stat, p_value = stats.ttest_ind(
        treatment_values, control_values, equal_var=False
    )
    t_stat = float(t_stat)
    p_value = float(p_value)
 
    # Calculate the Margin of Error
    margin = stats.t.ppf(1 - alpha / 2, dof) * se
 
    # Delta-Method Standard Error for the Relative Difference
    var_mean_treatment = var_treatment / n_treatment
    var_mean_control = var_control / n_control
    relative_se = np.sqrt(
        var_mean_treatment / mean_control ** 2
        + (mean_treatment ** 2 * var_mean_control) / mean_control ** 4
    )
    relative_margin = stats.t.ppf(1 - alpha / 2, dof) * relative_se

    # Return the Results
    return t_test_result(
        mean_control=mean_control,
        mean_treatment=mean_treatment,
        absolute_diff=absolute_diff,
        relative_diff=relative_diff,
        se=se,
        t_stat=t_stat,
        p_value=p_value,
        ci_low=absolute_diff - margin,
        ci_high=absolute_diff + margin,
        relative_ci_low=relative_diff - relative_margin,
        relative_ci_high=relative_diff + relative_margin,
    )

In [8]:
# Run the Test
result = t_test_binary(df, 
                metric="complete",
                assignment_col="assignment", 
                alpha=0.05)
print(result)

t_test_result(mean_control=0.8925925925925926, mean_treatment=0.7908158464289075, absolute_diff=-0.10177674616368515, relative_diff=-0.11402374051533191, se=np.float64(0.006144832246162851), t_stat=-16.561533972662026, p_value=7.89414530614067e-61, ci_low=np.float64(-0.11382179526821627), ci_high=np.float64(-0.08973169705915403), relative_ci_low=np.float64(-0.12663661036404272), relative_ci_high=np.float64(-0.10141087066662112))


# Check the Results against Scipy Functions

In [9]:
treatment = df.loc[df["assignment"], "complete"].to_numpy()
control = df.loc[~df["assignment"], "complete"].to_numpy()

res = stats.ttest_ind(treatment, control, equal_var=False)  # Welch's, matches your dof formula
ci = res.confidence_interval(confidence_level=0.95)
print(res.statistic, res.pvalue, res.df, ci.low, ci.high)

-16.561533972662026 7.89414530614067e-61 10418.649778600076 -0.11382284853228958 -0.08973064379508072
